# Analysis of Splicing Predictions on Assays

1. Liao et al. 2023: synthetic sequence splicing assay
2. Baeza-Centurion et al. 2025: *FAS* exon 6 mutagenesis assay
3. Chong et al. 2019: MFASS assay

## 1. Setup and Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Custom utility modules
import sequence_utils as seq_utils
import plotting_utils as plot_utils
import modeling_utils as model_utils

## 2. Data Loading

In [2]:
csv_paths = {
    "liao2023": "../data/Liao_2023_SplicingPredictions.csv.gz",
    "baeza-centurion2025": "../data/Baeza_Centurion_2025_SplicingPredictions.csv.gz",
    "chong2019": "../data/Chong_2019_SplicingPredictions.csv.gz",
}

display_names = {
    "liao2023": "Liao et al. 2023",
    "baeza-centurion2025": "FAS Exon 6",
    "chong2019": "MFASS",
}

assay_data = dict()
for key, path in csv_paths.items():
    assay_data[key] = pd.read_csv(path)

assay_data["liao2023"]["PSI"] = assay_data["liao2023"]["PSI"] * 100

assay_data["baeza-centurion2025"]["exon"] = assay_data["baeza-centurion2025"]["sequence"]
assay_data["baeza-centurion2025"]["PSI"] = assay_data["baeza-centurion2025"]["psi"]

# Filter MFASS only to WT and exonic variant sequences
assay_data["chong2019"] = assay_data["chong2019"].loc[(
    (assay_data["chong2019"]["label"] == "exon") |
    (assay_data["chong2019"]["category"] == "natural")
)]
assay_data["chong2019"]["PSI"] = assay_data["chong2019"]["v2_index"] * 100

## 3. Splicing Predictions

In [3]:
# Add average Pangolin tissue usage score
for key, data in assay_data.items():
    assay_data[key]["pangolin_average_usage_avg"] = (
        assay_data[key]["pangolin_heart_usage_avg"] +
        assay_data[key]["pangolin_liver_usage_avg"] +
        assay_data[key]["pangolin_brain_usage_avg"] +
        assay_data[key]["pangolin_testis_usage_avg"]
    ) / 4

In [4]:
assay_data["liao2023"][[
    "PSI", "spliceai_avg", "alphagenome_junctions_psi", 
    "pangolin_average_usage_avg",
]].corr()

,PSI,spliceai_avg,alphagenome_junctions_psi,pangolin_average_usage_avg
PSI,1.000000,0.846221,0.840621,0.780118
spliceai_avg,0.846221,1.000000,0.905043,0.933030
alphagenome_junctions_psi,0.840621,0.905043,1.000000,0.800085
pangolin_average_usage_avg,0.780118,0.933030,0.800085,1.000000


In [5]:
assay_data["baeza-centurion2025"][[
    "PSI", "spliceai_avg", "alphagenome_junctions_psi", 
    "pangolin_average_usage_avg",
]].corr()

,PSI,spliceai_avg,alphagenome_junctions_psi,pangolin_average_usage_avg
PSI,1.000000,0.830704,0.836509,0.752624
spliceai_avg,0.830704,1.000000,0.944121,0.785024
alphagenome_junctions_psi,0.836509,0.944121,1.000000,0.713120
pangolin_average_usage_avg,0.752624,0.785024,0.713120,1.000000


In [6]:
assay_data["chong2019"][[
    "PSI", "spliceai_avg", "alphagenome_junctions_psi", 
    "pangolin_average_usage_avg",
]].corr()

,PSI,spliceai_avg,alphagenome_junctions_psi,pangolin_average_usage_avg
PSI,1.000000,0.395731,0.452931,0.397659
spliceai_avg,0.395731,1.000000,0.792124,0.799672
alphagenome_junctions_psi,0.452931,0.792124,1.000000,0.804052
pangolin_average_usage_avg,0.397659,0.799672,0.804052,1.000000


## 4. Add CpG and stop codon metadata

In [7]:
for key, data in assay_data.items():
    assay_data[key] = seq_utils.add_CpG_features(assay_data[key], sequence_col="exon")
    assay_data[key] = seq_utils.add_stop_codon_features(assay_data[key], sequence_col="exon")

In [8]:
# Discretize CpG and GpC obs/exp ratios
for key, data in assay_data.items():
    for dinuc in ["CpG", "GpC"]:    
        assay_data[key][f"{dinuc}_obs_exp_disc"] = plot_utils.discretize_variable(
            assay_data[key][f"{dinuc}_obs_exp"], 
            0.25,
            min_val=None, 
            max_val=1.75,
            labels=True,
            include_above_max=True,
        )
        assay_data[key][f"{dinuc}_obs_exp_disc"] = plot_utils.discretize_variable(
            assay_data[key][f"{dinuc}_obs_exp"], 
            0.25,
            min_val=None, 
            max_val=1.75,
            labels=True,
            include_above_max=True,
        )

In [9]:
for key, data in assay_data.items():
    print(key, len(data))

liao2023 239772
baeza-centurion2025 5976
chong2019 16469


## 5. Add model errors

In [10]:
for key, data in assay_data.items():
    assay_data[key]["spliceai_avg_error"] = assay_data[key]["spliceai_avg"] * 100 - assay_data[key]["PSI"]
    assay_data[key]["alphagenome_junctions_psi_error"] = assay_data[key]["alphagenome_junctions_psi"] * 100 - assay_data[key]["PSI"]
    assay_data[key]["pangolin_average_usage_avg_error"] = assay_data[key]["pangolin_average_usage_avg"] * 100 - assay_data[key]["PSI"]

In [11]:
model_error_cols = {
    "spliceai_avg_error": "SpliceAI",
    "pangolin_average_usage_avg_error": "Pangolin",
    "alphagenome_junctions_psi_error": "AlphaGenome"
}

## 6. CpG Bias

In [21]:
bbox_to_anchors = {
    "liao2023" : (0.5, -0.55),
    "baeza-centurion2025": (0.5, -0.50),
    "chong2019": (0.5, -0.62),
}

ceiling_counts = {
    "liao2023" : 8,
    "baeza-centurion2025": 10,
    "chong2019": 8,
}

target_bins = {
    "liao2023" : 5,
    "baeza-centurion2025": 5,
    "chong2019": 10,
}

figsizes = {
    "liao2023" : (5.8, 3.25),
    "baeza-centurion2025": (5.8, 3.25),
    "chong2019": (5.8, 3.25),
}

In [22]:
group_col = {
    "liao2023" : "CpG_obs_exp_disc",
    "baeza-centurion2025": "CpG_obs",
    "chong2019": "CpG_obs_exp_disc",
}

group_continuous_col = {
    "liao2023" : "CpG_obs_exp",
    "baeza-centurion2025": "CpG_obs",
    "chong2019": "CpG_obs_exp",
}

group_label = {
    "liao2023" : "CpG Ratio",
    "baeza-centurion2025": "Number of CpG",
    "chong2019": "CpG Ratio",
}

for key, data in assay_data.items():
    for model_col, model_name in model_error_cols.items():
        fig = plot_utils.plot_grouped_violin_analysis(
            data,
            predictor_col=model_col,
            target_col="PSI",
            group_col=group_col[key],
            predictor_label=f"Model Error (Score - PSI)",
            target_label="Experimentally Measured PSI",
            group_label=group_label[key],
            target_bins=target_bins[key],
            ceiling_count=ceiling_counts[key],
            min_samples=20,
            title=f"{display_names[key]}, {model_name}",
            cmap='inferno',
            ylim=(-105, 105),
            legend=True,
            legend_loc="lower center",
            legend_ncols=8,
            bbox_to_anchor=bbox_to_anchors[key],
            dpi=600,
            figsize=figsizes[key],
            fontsize_title=10,
            fontsize_axis_labels=10,
            fontsize_xtick=10,
            fontsize_ytick=10,
            fontsize_legend=10,
            fontsize_sample_counts=5,
            numeric_binning=False,
            legend_row_major=True
        )
        fig.show()
        fig.savefig(f"figures/{key}_{model_col}_CpG_Error.svg", bbox_inches='tight')
        plt.close()
                
        model = smf.ols(f"{model_col} ~ {group_continuous_col[key]} + PSI", data=data).fit()
        print(key, model_name, model.summary())

liao2023 SpliceAI                             OLS Regression Results                            
Dep. Variable:     spliceai_avg_error   R-squared:                       0.462
Model:                            OLS   Adj. R-squared:                  0.462
Method:                 Least Squares   F-statistic:                 1.030e+05
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:23:48   Log-Likelihood:            -1.0174e+06
No. Observations:              239772   AIC:                         2.035e+06
Df Residuals:                  239769   BIC:                         2.035e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       9.5259      0.09

### GpC Control

In [23]:
group_col = {
    "liao2023" : "GpC_obs_exp_disc",
    "baeza-centurion2025": "GpC_obs",
    "chong2019": "GpC_obs_exp_disc",
}

group_continuous_col = {
    "liao2023" : "GpC_obs_exp",
    "baeza-centurion2025": "GpC_obs",
    "chong2019": "GpC_obs_exp",
}

group_label = {
    "liao2023" : "GpC Ratio",
    "baeza-centurion2025": "Number of GpC",
    "chong2019": "GpC Ratio",
}

for key, data in assay_data.items():
    for model_col, model_name in model_error_cols.items():
        fig = plot_utils.plot_grouped_violin_analysis(
            data,
            predictor_col=model_col,
            target_col="PSI",
            group_col=group_col[key],
            predictor_label=f"Model Error (Score - PSI)",
            target_label="Experimentally Measured PSI",
            group_label=group_label[key],
            target_bins=target_bins[key],
            ceiling_count=ceiling_counts[key],
            min_samples=20,
            title=f"{display_names[key]}, {model_name}",
            cmap='inferno',
            ylim=(-105, 105),
            legend=True,
            legend_loc="lower center",
            legend_ncols=8,
            bbox_to_anchor=bbox_to_anchors[key],
            dpi=600,
            figsize=figsizes[key],
            fontsize_title=10,
            fontsize_axis_labels=10,
            fontsize_xtick=10,
            fontsize_ytick=10,
            fontsize_legend=10,
            fontsize_sample_counts=5,
            numeric_binning=(key == "baeza-centurion2025"),
            numeric_bin_size=1,
            legend_row_major=True
        )
        fig.show()
        fig.savefig(f"figures/{key}_{model_col}_GpC_Error.svg", bbox_inches='tight')
        plt.close()

        model = smf.ols(f"{model_col} ~ {group_continuous_col[key]} + PSI", data=data).fit()
        print(key, model_name, model.summary())

liao2023 SpliceAI                             OLS Regression Results                            
Dep. Variable:     spliceai_avg_error   R-squared:                       0.419
Model:                            OLS   Adj. R-squared:                  0.419
Method:                 Least Squares   F-statistic:                 8.640e+04
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:23:53   Log-Likelihood:            -1.0267e+06
No. Observations:              239772   AIC:                         2.053e+06
Df Residuals:                  239769   BIC:                         2.053e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      21.5632      0.11

## 7. Stop Codon Bias

In [24]:
bbox_to_anchors = {
    "liao2023" : (0.5, -0.55),
}
ceiling_counts = {
    "liao2023" : 8,
}
target_bins = {
    "liao2023" : 5,
}
figsizes = {
    "liao2023" : (3.75, 2.5),#(2.8, 2.25),
}

stop_codon_cols = [
    ("Number of TAA", "TAA", "TAA_count"), 
    ("Number of TAG", "TAG", "TAG_count"),
    ("Number of TGA", "TGA", "TGA_count"),
    ("Number of Stop Codons", "Stop Codon", "stop_codon_count"),
]

for model_col, model_name in model_error_cols.items():
    for col_name, col_title, stop_codon_col in stop_codon_cols:
        fig = plot_utils.plot_grouped_violin_analysis(
            assay_data["liao2023"], 
            predictor_col=model_col,
            target_col="PSI", 
            group_col=stop_codon_col,
            group_label=col_name,
            predictor_label=f"Model Error (Score - PSI)",
            target_label="Experimentally Measured PSI",
            target_bins=target_bins["liao2023"],
            ceiling_count=ceiling_counts["liao2023"],
            title=f"Liao et al. 2023, {model_name} ({col_title})",
            cmap='inferno',
            ylim=(-105, 105),
            legend_loc="lower center",
            legend_ncols=5,
            bbox_to_anchor=bbox_to_anchors["liao2023"],
            dpi=600,
            figsize=figsizes["liao2023"],
            fontsize_title=10,
            fontsize_axis_labels=10,
            fontsize_ytick=10,
            fontsize_legend=10,
            fontsize_sample_counts=6.5,
            min_samples=20,
            numeric_binning=True,
            legend_row_major=True
        )
        fig.savefig(f"figures/liao2023_{model_col}_{stop_codon_col}_StopCodon_Error.svg", bbox_inches='tight')
        plt.close()

        model = smf.ols(f"{model_col} ~ {stop_codon_col} + PSI", data=assay_data["liao2023"]).fit()
        print("liao2023", model_name, model.summary())

liao2023 SpliceAI                             OLS Regression Results                            
Dep. Variable:     spliceai_avg_error   R-squared:                       0.438
Model:                            OLS   Adj. R-squared:                  0.438
Method:                 Least Squares   F-statistic:                 9.354e+04
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:24:00   Log-Likelihood:            -1.0226e+06
No. Observations:              239772   AIC:                         2.045e+06
Df Residuals:                  239769   BIC:                         2.045e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     25.8920      0.085  

## 8. Secondary Structure MFE (Minimum Free Energy) Analysis

In [25]:
# Discretize MFE
assay_data["liao2023"]["predicted_MFE_disc"] = plot_utils.mfe_discretize_variable(
    assay_data["liao2023"]["predicted_MFE"], 
    5, 
    min_val=-30, 
    max_val=-10, 
    labels=True,
    include_above_max=True,
    include_below_min=True,
)

In [26]:
bbox_to_anchors = {
    "liao2023" : (0.5, -0.55),
}
ceiling_counts = {
    "liao2023" : 8,
}
target_bins = {
    "liao2023" : 5,
}
figsizes = {
    "liao2023" : (2.8, 2.25),
}

for key in ["liao2023"]:
    for model_col, model_name in model_error_cols.items():
        fig = plot_utils.plot_grouped_violin_analysis(
            assay_data[key],
            predictor_col=model_col,
            target_col="PSI",
            group_col="predicted_MFE_disc",
            predictor_label=f"Model Error (Score - PSI)",
            target_label="Experimentally Measured PSI",
            group_label="Predicted Minimum Free Energy (kcal/mol) (↓ = More Stable)",
            target_bins=target_bins[key],
            ceiling_count=ceiling_counts[key],
            min_samples=20,
            title=f"{display_names[key]}, {model_name}",
            cmap='inferno',
            ylim=(-105, 105),
            legend=True,
            legend_loc="lower center",
            legend_ncols=8,
            bbox_to_anchor=bbox_to_anchors[key],
            dpi=600,
            figsize=figsizes[key],
            fontsize_title=8,
            fontsize_axis_labels=7,
            fontsize_xtick=7,
            fontsize_ytick=7,
            fontsize_legend=7,
            fontsize_sample_counts=5,
            numeric_binning=False,
            legend_row_major=True,
            group_orders=[
                '< -30',
                '(-30, -25]',
                '(-25, -20]',
                '(-20, -15]',
                '(-15, -10]',
                '-10+'
            ]
        )
        fig.show()
        fig.savefig(f"figures/{key}_{model_col}_MFE_Error.svg", bbox_inches='tight')
        plt.close()
                
        model = smf.ols(f"{model_col} ~ predicted_MFE + PSI", data=assay_data[key]).fit()
        print(key, model_name, model.summary())

liao2023 SpliceAI                             OLS Regression Results                            
Dep. Variable:     spliceai_avg_error   R-squared:                       0.461
Model:                            OLS   Adj. R-squared:                  0.461
Method:                 Least Squares   F-statistic:                 1.025e+05
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        01:24:30   Log-Likelihood:            -1.0176e+06
No. Observations:              239772   AIC:                         2.035e+06
Df Residuals:                  239769   BIC:                         2.035e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         6.3562    